In [7]:
import duckdb
import pandas as pd

train_path = "../data/train.parquet"

# Cap DuckDB's RAM use and let it spill to disk instead of crashing
con = duckdb.connect()
con.sql("SET memory_limit = '4GB'")

print("=== STEP 2: EXPLORE VIA DUCKDB ===")

# 1. Basic summary (aggregates only - nothing large is pulled into pandas)
summary_query = f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS total_subjects,
        MIN(date) AS start_time,
        MAX(date) AS end_time
    FROM '{train_path}'
"""
print(con.sql(summary_query).df())

# 2. Breakdown of insulin delivery modalities
modality_query = f"""
    SELECT insulin_delivery_modality, COUNT(DISTINCT id) AS unique_patients, COUNT(*) AS row_count
    FROM '{train_path}'
    GROUP BY insulin_delivery_modality
    ORDER BY row_count DESC
"""
print("\n--- Modality Breakdown ---")
print(con.sql(modality_query).df())

# NOTE: Filtering is done in the next cell by streaming to disk.
# Never SELECT * the full table into pandas - it is ~130M rows x 37 columns.

=== STEP 2: EXPLORE VIA DUCKDB ===
   total_rows  total_subjects          start_time            end_time
0   132811004            1188 2011-11-17 21:00:00 2027-07-14 15:30:00

--- Modality Breakdown ---
  insulin_delivery_modality  unique_patients  row_count
0                       AID             1114  112006707
1                       SAP              226   18998161
2                       NaN               20    1164394
3                       MDI               24     641742


In [8]:
import duckdb

train_path = "../data/train.parquet"
output_path = "../data/train_filtered.parquet"

con = duckdb.connect()
con.sql("SET memory_limit = '4GB'")       # spill to disk instead of OOM
con.sql("SET preserve_insertion_order = false")

print("=== STEP 3: FILTER (Pumps Only) - streaming to disk ===")

# - Keep only confirmed pump users (AID / SAP). Rows with unknown (NULL)
#   modality are dropped because they could be MDI users.
# - Keep only the columns the model needs.
# - Collapse duplicate (id, date) rows into one reading per 5-min slot.
# - Sort by id, date so each patient's rows are contiguous on disk.
con.sql(f"""
    COPY (
        SELECT 
            id,
            date,
            AVG(CGM)     AS CGM,
            MAX(insulin) AS insulin,
            MAX(carbs)   AS carbs,
            ANY_VALUE(insulin_delivery_modality) AS insulin_delivery_modality
        FROM '{train_path}'
        WHERE insulin_delivery_modality IN ('AID', 'SAP')
        GROUP BY id, date
        ORDER BY id, date
    ) TO '{output_path}' (FORMAT PARQUET);
""")

print(con.sql(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS subjects,
           COUNT(*) FILTER (WHERE CGM IS NULL) AS cgm_missing
    FROM '{output_path}'
""").df())
print("Successfully saved clean dataset to:", output_path)

=== STEP 3: FILTER (Pumps Only) - streaming to disk ===
        rows  subjects  cgm_missing
0  123613512      1165     19046339
Successfully saved clean dataset to: ../data/train_filtered.parquet


In [9]:
import duckdb
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

input_path = "../data/train_filtered.parquet"

WINDOW_SIZE = 24   # 24 x 5 min = 2 hours of history
HORIZON = 6        # 6 x 5 min = 30 minutes ahead
STEP_MIN = 5       # expected sampling interval (minutes)
FEATURES = ["CGM", "insulin", "carbs"]


def build_sliding_windows(parquet_file_path, max_patients=5):
    """
    Generator that creates sliding windows one whole patient at a time.

    Input  X: last 2 hours of [CGM, insulin, carbs] -> shape (24, 3)
    Target y: CGM exactly 30 minutes after the last input step

    A window is only produced from a run of rows that are exactly 5 min apart
    with no missing CGM, so 24 steps always means 2 hours and the target is
    always exactly 30 minutes ahead.
    """
    con = duckdb.connect()
    con.sql("SET memory_limit = '4GB'")

    # Query each patient separately so no patient is ever split across batches
    patient_ids = [r[0] for r in con.execute(
        f"SELECT DISTINCT id FROM '{parquet_file_path}' ORDER BY id"
    ).fetchall()]

    for patient_id in patient_ids[:max_patients]:
        group = con.execute(
            f"SELECT date, CGM, insulin, carbs FROM '{parquet_file_path}' "
            f"WHERE id = ? ORDER BY date",
            [patient_id],
        ).df()

        # Missing insulin / carbs means "none recorded" -> 0.
        group[["insulin", "carbs"]] = group[["insulin", "carbs"]].fillna(0.0)

        # Rows without a CGM reading can't be used as inputs or targets.
        # Dropping them creates a >5 min gap, which breaks the sequence below.
        group = group.dropna(subset=["CGM"]).reset_index(drop=True)
        if len(group) < WINDOW_SIZE + HORIZON:
            continue

        # Start a new sequence whenever the step isn't exactly 5 minutes
        time_diff = group["date"].diff().dt.total_seconds() / 60.0
        group["sequence_id"] = (time_diff != STEP_MIN).cumsum()

        for _, seq in group.groupby("sequence_id"):
            n = len(seq)
            if n < WINDOW_SIZE + HORIZON:
                continue

            values = seq[FEATURES].to_numpy(dtype=np.float32)   # (n, 3)
            cgm = values[:, 0]

            # All windows at once: (n_windows, 24, 3)
            n_windows = n - WINDOW_SIZE - HORIZON + 1
            X = sliding_window_view(values, WINDOW_SIZE, axis=0)[:n_windows]
            X = X.transpose(0, 2, 1)
            y = cgm[WINDOW_SIZE + HORIZON - 1 : WINDOW_SIZE + HORIZON - 1 + n_windows]

            for x_window, y_target in zip(X, y):
                yield patient_id, x_window, y_target


print("Window generator defined successfully!")

print("\nGenerating sample windows for the first few patients:")
generator = build_sliding_windows(input_path, max_patients=3)

sample_count = 0
for pid, x_window, y_target in generator:
    if sample_count < 3:
        print(f"Patient ID: {pid} | Input window shape: {x_window.shape} "
              f"(CGM, insulin, carbs) | Target 30-min CGM: {y_target}")
    sample_count += 1

print(f"Total sample windows generated from test subset: {sample_count}")

Window generator defined successfully!

Generating sample windows for the first few patients:
Patient ID: 1 | Input window shape: (24, 3) (CGM, insulin, carbs) | Target 30-min CGM: 99.0
Patient ID: 1 | Input window shape: (24, 3) (CGM, insulin, carbs) | Target 30-min CGM: 113.0
Patient ID: 1 | Input window shape: (24, 3) (CGM, insulin, carbs) | Target 30-min CGM: 111.0
Total sample windows generated from test subset: 585534


In [ ]:
#to visulaise a small part of the train filtered data 
import duckdb

duckdb.sql("""
    COPY (
        SELECT source_file || '_' || id AS patient_id, date, CGM, insulin, carbs,
               insulin_delivery_modality, source_file
        FROM '../data/train.parquet'
        WHERE (source_file, id) IN (
            SELECT DISTINCT source_file, id FROM '../data/train.parquet'
            WHERE insulin_delivery_modality IN ('AID', 'SAP')
            USING SAMPLE 5 ROWS              -- number of patients
        )
        ORDER BY patient_id, date
    ) TO '../data/sample_5_patients.csv' (HEADER, DELIMITER ',');
""")

duckdb.sql("""
    COPY (SELECT * FROM '../data/train.parquet' LIMIT 10000)
    TO '../data/sample_first_10k.csv' (HEADER);
""")


sample_5_patients_df = pd.read_csv('../data/sample_5_patients.csv', parse_dates=['date'])
sample_5_patients_df.head(10)


,patient_id,date,CGM,insulin,carbs,insulin_delivery_modality,source_file
0,OhioT1DM_584,2025-05-14 00:05:00,48.0,0.145833,NaN,SAP,OhioT1DM
1,OhioT1DM_584,2025-05-14 00:10:00,48.0,0.145833,NaN,SAP,OhioT1DM
2,OhioT1DM_584,2025-05-14 00:15:00,53.0,0.145833,NaN,SAP,OhioT1DM
3,OhioT1DM_584,2025-05-14 00:20:00,63.0,0.145833,NaN,SAP,OhioT1DM
4,OhioT1DM_584,2025-05-14 00:25:00,69.0,0.145833,NaN,SAP,OhioT1DM
5,OhioT1DM_584,2025-05-14 00:30:00,74.0,0.145833,NaN,SAP,OhioT1DM
6,OhioT1DM_584,2025-05-14 00:35:00,77.0,0.145833,NaN,SAP,OhioT1DM
7,OhioT1DM_584,2025-05-14 00:40:00,70.0,0.000000,NaN,SAP,OhioT1DM
8,OhioT1DM_584,2025-05-14 00:45:00,59.0,0.000000,NaN,SAP,OhioT1DM
9,OhioT1DM_584,2025-05-14 00:50:00,47.0,0.000000,NaN,SAP,OhioT1DM


In [14]:
sample_first_10k_df = pd.read_csv('../data/sample_first_10k.csv', parse_dates=['date'])
sample_first_10k_df.head(10)


,CGM,age,air_temp,basal,bolus,calories_burned,carbs,cgm_device,misc_notes,date,...,workout_duration,workout_intensity,workout_label,treatment_group,randomization_date,extension_date,gender,age_of_diagnosis,ethnicity,subject_split_across_traintest
0,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:20:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
1,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:25:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
2,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:30:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
3,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:35:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
4,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:40:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
5,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:45:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
6,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:50:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
7,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 09:55:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
8,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 10:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
9,NaN,NaN,NaN,0.05,NaN,NaN,NaN,Medtronic Enlite,NaN,2027-04-07 10:05:00,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,NaN,NaN,False
